In [0]:
from pyspark.sql.functions import current_timestamp

def add_ingestion_date(df):
  return df.withColumn('ingestion_date', current_timestamp())

In [0]:
def delete_table_exists(v_file_date, v_db, v_table, v_column):
    if spark.catalog.tableExists(f"{v_db}.{v_table}"):
        spark.sql(f"DELETE FROM {v_db}.{v_table} WHERE '{v_column}' = '{v_file_date}'")

In [0]:
from delta.tables import DeltaTable

def merge_delta_lake( input_df, db_name, table_name, merge_condition, partition_column ):
    if spark.catalog.tableExists(f"{db_name}.{table_name}"):

        deltaTable = DeltaTable.forName(spark, f"{db_name}.{table_name}")

        deltaTable.alias('tgt') \
        .merge(
            input_df.alias('src'),
            merge_condition
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()

    else:
        input_df.write.mode("overwrite").partitionBy(f"{partition_column}").format("delta").saveAsTable(f"{db_name}.{table_name}")

    